# BeamFinder — Training Notebook

Fine-tunes YOLO26s on the DeepSense Scenario 23 drone dataset.

**Setup:** Runtime → Change runtime type → **T4 GPU** (or A100 if available on Pro).

In [ ]:
# Install dependencies (kernel may restart — this is normal, just continue to the next cell)
!pip install -q ultralytics

In [ ]:
# Clone repo and cd into it
import os

if not os.path.exists("beamfinder"):
    !git clone https://github.com/seriouslysahid/beamfinder.git
else:
    !cd beamfinder && git pull

%cd beamfinder

In [ ]:
# Verify GPU
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Train
from ultralytics import YOLO

model = YOLO("yolo26s.pt")

model.train(
    data="data.yaml",
    epochs=100,
    imgsz=960,
    batch=0.85,
    patience=20,
    cache="disk",
    workers=4,
    cos_lr=True,
    deterministic=False,
    project="runs",
    name="drone_detect",
    exist_ok=True,
    rect=True,
    save_period=10,
    degrees=15.0,
    flipud=0.5,
    scale=0.9,
    translate=0.2,
)

In [ ]:
# Validate on val and test splits
metrics = model.val(imgsz=960)
print(f"Val  — mAP50: {metrics.box.map50:.4f}  mAP50-95: {metrics.box.map:.4f}")

test_metrics = model.val(split="test", imgsz=960)
print(f"Test — mAP50: {test_metrics.box.map50:.4f}  mAP50-95: {test_metrics.box.map:.4f}")

In [ ]:
# Show training plots
from IPython.display import Image, display
from pathlib import Path

run_dir = Path("runs/drone_detect")
for plot in ["results.png", "confusion_matrix.png", "F1_curve.png", "PR_curve.png"]:
    p = run_dir / plot
    if p.exists():
        print(f"\n{plot}")
        display(Image(filename=str(p), width=800))

In [ ]:
# Save weights to Google Drive (optional)
from google.colab import drive
import shutil

drive.mount("/content/drive")
save_dir = Path("/content/drive/MyDrive/beamfinder_weights")
save_dir.mkdir(parents=True, exist_ok=True)

for w in ["best.pt", "last.pt"]:
    src = run_dir / "weights" / w
    if src.exists():
        shutil.copy2(src, save_dir / w)
        print(f"Saved {w} -> {save_dir / w}")